In [123]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas scikit-learn matplotlib seaborn -q

import platform
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# 재현성: 같은 난수를 항상 같게 만듭니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록)
system = platform.system()
if system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.4.6
pandas: 3.0.3


In [183]:
# ─────────────────────────────────────────────
# 모두마켓 이번 달 주문 데이터 생성 — 자기완결적 스냅샷
# (지난 노드에서 다룬 오염 요소를 적당히 섞어 둡니다)
# ─────────────────────────────────────────────
np.random.seed(42)
n = 1500

regions = np.random.choice(["서울", "경기", "부산", "인천", "대구"], n, p=[0.4, 0.25, 0.15, 0.1, 0.1])
membership = np.random.choice(["basic", "silver", "gold", "vip"], n, p=[0.5, 0.25, 0.15, 0.1])
channels = np.random.choice(["web", "app", "app ", "APP"], n, p=[0.5, 0.4, 0.05, 0.05])
categories = np.random.choice(["패션", "뷰티", "식품", "가전", "도서"], n)

prices = np.random.choice([9900, 19900, 29900, 49900, 89900, 129900, 249900], n,
                          p=[0.2, 0.25, 0.2, 0.15, 0.1, 0.06, 0.04])
quantities = np.random.choice([1, 1, 1, 2, 2, 3], n)
amount = (prices * quantities).astype(float)

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n + 1)],
    "customer_age": np.random.normal(35, 9, n).round().astype(int),
    "region": regions,
    "membership": membership,
    "channel": channels,
    "category": categories,
    "price": prices.astype(float),
    "quantity": quantities,
    "amount": amount,
})

# 오염 심기: 결측·이상치·표기 혼재
orders.loc[np.random.choice(n, 60, replace=False), "amount"] = np.nan
orders.loc[np.random.choice(n, 30, replace=False), "customer_age"] = np.nan
orders.loc[5, "customer_age"] = 999             # 입력 실수성 이상치
orders.loc[8, "customer_age"] = -3              # 불가능한 음수
orders.loc[12, "quantity"] = 80                 # 비정상적으로 큰 주문
orders.loc[20, "region"] = " 서울 "             # 앞뒤 공백
orders.loc[21, "region"] = "Seoul"              # 영문 표기
orders.loc[40, "membership"] = "VIP"            # 대소문자 혼재

print("이번 달 주문 데이터 준비 완료:", orders.shape)
orders.head()

이번 달 주문 데이터 준비 완료: (1500, 9)


,order_id,customer_age,region,membership,channel,category,price,quantity,amount
0,O00001,30.0,서울,silver,app,패션,19900.0,1,NaN
1,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0
2,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0
3,O00004,24.0,경기,basic,app,뷰티,19900.0,1,NaN
4,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0


In [7]:
# ─────────────────────────────────────────────
# 시나리오 0 — 원본 CSV 파일 준비 (실무에서는 외부에서 받아오는 단계)
# ─────────────────────────────────────────────
from pathlib import Path

work_dir = Path("d006_work")
work_dir.mkdir(exist_ok=True)
input_csv  = work_dir / "orders_raw.csv"
output_csv = work_dir / "orders_clean.csv"

orders.to_csv(input_csv, index=False)
print("원본 CSV 저장:", input_csv, "—", input_csv.stat().st_size, "bytes")

원본 CSV 저장: d006_work\orders_raw.csv — 81901 bytes


In [184]:
#1 orders_clean copy 생성 함수
def orders_copy(orders):
    orders_clean = orders.copy()
    return orders_clean

orders_clean = orders_copy(orders)

In [185]:
orders_clean.head()

,order_id,customer_age,region,membership,channel,category,price,quantity,amount
0,O00001,30.0,서울,silver,app,패션,19900.0,1,NaN
1,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0
2,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0
3,O00004,24.0,경기,basic,app,뷰티,19900.0,1,NaN
4,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0


In [41]:
#2 customoer_age 전처리
def age_clean(orders_clean):
    orders_clean["customer_age"] = orders_clean["customer_age"].where(
        (orders_clean["customer_age"] >= 0) & (orders_clean["customer_age"] <= 120), 
        np.nan)
    orders_clean["customer_age"] = orders_clean["customer_age"].astype("int64")
    return orders_clean

orders_clean = age_clean(orders_clean)

orders_clean.head()

,order_id,customer_age,region,membership,channel,category,price,quantity,amount
0,O00001,30,서울,silver,app,패션,19900.0,1,NaN
1,O00002,24,대구,basic,app,가전,249900.0,3,749700.0
2,O00003,45,부산,basic,web,패션,19900.0,1,19900.0
3,O00004,24,경기,basic,app,뷰티,19900.0,1,NaN
4,O00005,41,서울,basic,app,패션,19900.0,1,19900.0


In [43]:
orders_clean["region"].value_counts(dropna=False)

region
서울       608
경기       361
부산       218
인천       158
대구       153
 서울        1
Seoul      1
Name: count, dtype: int64

In [151]:
# region 전치리 함수

def region_clean(orders_clean):
    orders_clean["region"] = (
        orders_clean["region"].str.strip()
        .str.lower()
        .replace({"seoul" : "서울"}))
    return orders_clean

orders_clean = region_clean(orders_clean)
orders_clean["region"].value_counts(dropna=False)

region
서울    610
경기    361
부산    218
인천    158
대구    153
Name: count, dtype: int64

In [76]:
# membership 전처리 함수

def membership_clean(orders_clean):
    orders_clean["membership"] = orders_clean["membership"].str.lower()
    return orders_clean

orders_clean = membership_clean(orders_clean)

orders_clean["membership"].value_counts(dropna = False)

membership
basic     744
silver    374
gold      236
vip       146
Name: count, dtype: int64

In [80]:
# channel 전처리 함수

def channel_clean(orders_clean):
    orders_clean["channel"] = (orders_clean["channel"].str.strip().str.lower())
    return orders_clean

orders_clean = channel_clean(orders_clean)

orders_clean["channel"].value_counts(dropna = False)

channel
web    762
app    738
Name: count, dtype: int64

In [81]:
orders_clean["category"].value_counts(dropna = False)

category
뷰티    340
도서    310
패션    295
가전    284
식품    271
Name: count, dtype: int64

In [82]:
orders_clean["price"].value_counts(dropna = False)

price
19900.0     377
29900.0     307
9900.0      306
49900.0     200
89900.0     148
129900.0    106
249900.0     56
Name: count, dtype: int64

In [83]:
orders_clean["quantity"].value_counts(dropna = False)

quantity
1     794
2     483
3     222
80      1
Name: count, dtype: int64

In [120]:
#region encoding(one-hot)
def region_encoding(orders_clean):
    region_dummies = pd.get_dummies(orders_clean["region"], prefix="region", dtype=int)
    orders_clean = pd.concat([orders_clean.drop(columns=["region"]), region_dummies], axis=1)
    return orders_clean


In [128]:
#membership encoding (ordinal)
def membership_encoding(orders_clean):
    orders_clean["membership"] = orders_clean["membership"].map({"basic": 0, "silver": 1, "gold" : 2, "vip" : 3})
    return orders_clean


In [130]:
#category encoding (one-hot) 0/1

def category_encoding(orders_clean):
    category_dummies = pd.get_dummies(orders_clean["category"], prefix="category", dtype=int)
    orders_clean = pd.concat([orders_clean.drop(columns=["category"]), category_dummies], axis=1)
    return orders_clean


In [131]:
#Channel encoding (label) 0/1
def channel_encoding(orders_clean):
    orders_clean["channel"] = orders_clean["channel"].map({"app": 0, "web": 1})
    return orders_clean

In [122]:
orders_clean["membership"].value_counts()

membership
basic     744
silver    374
gold      236
vip       145
VIP         1
Name: count, dtype: int64

In [138]:
import joblib
from sklearn.preprocessing import RobustScaler

In [140]:
#customer_age scaling (robust)

def age_scaling(orders_clean, age_scaler_path="age_scaler.pkl"):
    if os.path.exists(age_scaler_path):
        # 이미 저장된 규칙이 있으면 -> 불러와서 재사용
        age_scaler = joblib.load(age_scaler_path)
        orders_clean["customer_age_scaled"] = age_scaler.transform(orders_clean[["customer_age"]])
    else:
        # 저장된 규칙이 없으면 -> 처음이니 새로 학습해서 저장
        age_scaler = RobustScaler()
        orders_clean["customer_age_scaled"] = age_scaler.fit_transform(orders_clean[["customer_age"]])
        joblib.dump(age_scaler, age_scaler_path)
    
    return orders_clean

In [141]:
#price scaling (robust)

def price_scaling(orders_clean, price_scaler_path="price_scaler.pkl"):
    if os.path.exists(price_scaler_path):
        price_scaler = joblib.load(price_scaler_path)
        orders_clean["price_scaled"] = price_scaler.transform(orders_clean[["price"]])
    else:
        price_scaler = RobustScaler()
        orders_clean["price_scaled"] = price_scaler.fit_transform(orders_clean[["price"]])
        joblib.dump(price_scaler, price_scaler_path)
    
    return orders_clean

In [142]:
#quantity scaling (robust)

def quantity_scaling(orders_clean, quantity_scaler_path="quantity_scaler.pkl"):
    if os.path.exists(quantity_scaler_path):
        quantity_scaler = joblib.load(quantity_scaler_path)
        orders_clean["quantity_scaled"] = quantity_scaler.transform(orders_clean[["quantity"]])
    else:
        quantity_scaler = RobustScaler()
        orders_clean["quantity_scaled"] = quantity_scaler.fit_transform(orders_clean[["quantity"]])
        joblib.dump(quantity_scaler, quantity_scaler_path)
    
    return orders_clean

In [143]:
#amount scaling (robust)

def amount_scaling(orders_clean, amount_scaler_path="amount_scaler.pkl"):
    if os.path.exists(amount_scaler_path):
        amount_scaler = joblib.load(amount_scaler_path)
        orders_clean["amount_scaled"] = amount_scaler.transform(orders_clean[["amount"]])
    else:
        amount_scaler = RobustScaler()
        orders_clean["amount_scaled"] = amount_scaler.fit_transform(orders_clean[["amount"]])
        joblib.dump(amount_scaler, amount_scaler_path)
    
    return orders_clean

In [157]:
import os

In [186]:
#최종 method chaining

def preprocess(orders, verbose=True):
    
    def log_step(data, step_name):
        if verbose:
            print(f"[{step_name}] 행 수: {len(data)}")
        return data
    
    orders_clean = (
        orders
            .pipe(orders_copy).pipe(log_step, "orders_copy") # orders_clean copy 생성 함수
            .pipe(age_clean).pipe(log_step, "age_clean") # customoer_age 전처리
            .pipe(membership_clean).pipe(log_step, "membership_clean") # membership 전처리 함수
            .pipe(channel_clean).pipe(log_step, "channel_clean") # channel 전처리 함수
            .pipe(region_clean).pipe(log_step, "region_clean") # region 전치리 함수
            .pipe(region_encoding).pipe(log_step, "region_encoding") #region encoding(one-hot)
            .pipe(membership_encoding).pipe(log_step, "membership_encoding") #membership encoding (ordinal)
            .pipe(channel_encoding).pipe(log_step, "channel_encoding") #Channel encoding (label) 0/1
            .pipe(category_encoding).pipe(log_step, "category_encoding") #category encoding (one-hot)
            .pipe(age_scaling).pipe(log_step, "age_scaling") #customer_age scaling (robust)
            .pipe(price_scaling).pipe(log_step, "price_scaling") #price scaling (robust)
            .pipe(quantity_scaling).pipe(log_step, "quantity_scaling") #quantity scaling (robust)
            .pipe(amount_scaling).pipe(log_step, "amount_scaling") #amount scaling (robust)
    )
    return orders_clean

orders_clean = preprocess(orders)

[orders_copy] 행 수: 1500
[age_clean] 행 수: 1500
[membership_clean] 행 수: 1500
[channel_clean] 행 수: 1500
[region_clean] 행 수: 1500
[region_encoding] 행 수: 1500
[membership_encoding] 행 수: 1500
[channel_encoding] 행 수: 1500
[category_encoding] 행 수: 1500
[age_scaling] 행 수: 1500
[price_scaling] 행 수: 1500
[quantity_scaling] 행 수: 1500
[amount_scaling] 행 수: 1500


In [188]:
print("===== 데이터 정제 리포트 =====")
print(f"▸ 입력 행 수: {len(orders):,}")
print(f"▸ 출력 행 수: {len(orders_clean):,}")
print(f"▸ 제거된 비율: {(len(orders)-len(orders_clean))/len(orders)*100:.2f}%")
print()
print("▸ 컬럼별 결측 비율(상위 5개)")
print(orders_clean.isna().mean().mul(100).sort_values(ascending=False).head(5).round(1))
print()
new_cols = set(orders_clean.columns) - set(orders.columns)
# 1) 새로 생긴 컬럼 (One-Hot 등)
new_columns = set(orders_clean.columns) - set(orders.columns)

# 2) 기존 컬럼인데, 문자 -> 숫자로 바뀐 경우 (Label/Ordinal 등)
common_columns = set(orders.columns) & set(orders_clean.columns)
inplace_encoded = [
    col for col in common_columns
    if not pd.api.types.is_numeric_dtype(orders[col]) 
    and pd.api.types.is_numeric_dtype(orders_clean[col])
]

total_encoded = len(new_columns) + len(inplace_encoded)

print(f"▸ 새로 만들어진 파생/인코딩 컬럼 수: {total_encoded}")
print(f"   - 신규 생성: {sorted(new_columns)}")
print(f"   - 기존 컬럼 인코딩: {inplace_encoded}")
print()
print("▸ 자료형 분포 요약")
print(orders_clean.dtypes.value_counts())

===== 데이터 정제 리포트 =====
▸ 입력 행 수: 1,500
▸ 출력 행 수: 1,500
▸ 제거된 비율: 0.00%

▸ 컬럼별 결측 비율(상위 5개)
amount                 4.0
amount_scaled          4.0
customer_age_scaled    2.1
customer_age           2.1
order_id               0.0
dtype: float64

▸ 새로 만들어진 파생/인코딩 컬럼 수: 16
   - 신규 생성: ['amount_scaled', 'category_가전', 'category_도서', 'category_뷰티', 'category_식품', 'category_패션', 'customer_age_scaled', 'price_scaled', 'quantity_scaled', 'region_경기', 'region_대구', 'region_부산', 'region_서울', 'region_인천']
   - 기존 컬럼 인코딩: ['membership', 'channel']

▸ 자료형 분포 요약
int64      13
float64     6
str         1
Int64       1
Name: count, dtype: int64


In [193]:
from pathlib import Path

work_dir = Path("D006_work")
work_dir.mkdir(exist_ok=True)
output_csv = work_dir / "orders_clean.csv"
orders_clean.to_csv(output_csv, index=False, encoding="utf-8-sig")
print("저장 완료:", output_csv)

저장 완료: D006_work\orders_clean.csv
